# Module 5.4 — Asyncio Foundations

### Python Mastery · Track 5: Concurrency & Asynchronous Programming

Asynchronous programming is a third model of concurrency, alongside threads and processes. A single thread runs many tasks by switching between them at well-defined pause points, achieving high concurrency for input/output-bound work with low overhead. This module introduces coroutines, the event loop, awaiting, and running tasks concurrently.

**Important: running async code in a notebook.** A notebook already has a running event loop, so the usual script entry point `asyncio.run(...)` does **not** work in a cell; it raises an error. Inside a notebook you instead use `await` directly at the top level of a cell, which this notebook does throughout. The difference is explained in Part 1, and the script form is shown so you know what to write in a `.py` file.

### Learning objectives

After completing this module you will be able to:

1. Define coroutines with `async def` and pause them with `await`.
2. Explain the event loop and how it interleaves tasks.
3. Run coroutines concurrently with `asyncio.gather`.
4. Create and manage tasks with `asyncio.create_task`.
5. Distinguish how async code is started in a script versus a notebook.

**Prerequisites:** Tracks 1 to 4, and Modules 5.1 to 5.3.

---

## Part 1 · Coroutines, `await`, and How to Run Them

A function defined with `async def` is a **coroutine**. Calling it does not run it; it returns a coroutine object that must be driven by an event loop. Inside a coroutine, `await` pauses execution until an awaited operation completes, handing control back to the loop so other work can proceed in the meantime.

How you start the loop depends on where you are:

- In a `.py` **script**, the entry point is `asyncio.run(main())`, which creates a loop, runs the coroutine, and closes the loop.
- In a **notebook**, a loop is already running, so `asyncio.run` fails. You simply `await` the coroutine at the top of a cell.

The cell below uses the notebook form.

In [ ]:
import asyncio

async def say(message, delay):
    await asyncio.sleep(delay)        # pause without blocking the whole program
    return f"{message} (after {delay}s)"

# In a notebook we await directly. (In a script you would write: asyncio.run(say(...)).)
result = await say("hello", 0.05)
print(result)

To make the script-versus-notebook distinction concrete, here is the same logic written as a standalone script. Run it with `!python` to see the `asyncio.run` form working in its proper context.

In [ ]:
%%writefile async_script.py
import asyncio

async def main():
    await asyncio.sleep(0.05)
    print("hello from a script using asyncio.run")

if __name__ == "__main__":
    asyncio.run(main())              # the correct entry point in a .py file

In [ ]:
!python async_script.py

## Part 2 · The Event Loop and Cooperative Switching

The event loop runs one coroutine until it hits an `await` that cannot complete immediately (such as waiting on a timer or network). At that point the coroutine pauses and the loop runs another ready coroutine. This is **cooperative** concurrency: tasks voluntarily yield at `await` points. Because only one thing runs at a time, there are no data races of the kind seen with threads, but a coroutine that never awaits will block everything.

In [ ]:
import asyncio

events = []

async def task(name, steps):
    for i in range(steps):
        events.append(f"{name} step {i}")
        await asyncio.sleep(0.01)     # yield to the loop after each step

# Run two tasks concurrently; their steps interleave at the await points.
await asyncio.gather(task("A", 3), task("B", 3))

print("interleaved execution:")
for e in events:
    print(" ", e)

## Part 3 · Running Coroutines Concurrently with `gather`

`asyncio.gather` runs several coroutines concurrently and waits for them all, returning their results **in the order you passed them**, regardless of which finished first. This is the async counterpart to running tasks in parallel, and it is where async shines: many overlapping waits handled by one thread.

In [ ]:
import asyncio
import time

async def fetch(item, delay):
    await asyncio.sleep(delay)        # simulate a network request
    return f"data for {item}"

start = time.perf_counter()

# All three run concurrently; total time is the longest, not the sum.
results = await asyncio.gather(
    fetch("a", 0.05),
    fetch("b", 0.05),
    fetch("c", 0.05),
)

elapsed = time.perf_counter() - start
print("results (in passed order):", results)
print(f"took about {elapsed:.2f}s, roughly one delay, not three")

## Part 4 · Tasks with `create_task`

`asyncio.create_task` schedules a coroutine to run **as soon as possible**, returning a `Task` you can await later. This lets you start work in the background and continue, then collect the result when you need it. Awaiting a task that has already finished returns its result immediately.

In [ ]:
import asyncio

async def background_work(n):
    await asyncio.sleep(0.03)
    return n * 10

# Start the task now; it begins running in the background.
task = asyncio.create_task(background_work(5))

# Do other things meanwhile.
print("doing other work while the task runs...")
await asyncio.sleep(0.01)
print("still working...")

# Now wait for the task's result.
value = await task
print("task result:", value)

Creating several tasks and then awaiting them all is a common pattern, equivalent in effect to `gather` but giving you individual task handles to inspect or cancel.

In [ ]:
import asyncio

async def job(n):
    await asyncio.sleep(0.02)
    return n * n

tasks = [asyncio.create_task(job(n)) for n in range(1, 5)]   # all start running
results = [await t for t in tasks]                            # collect each result
print("results:", results)

## Part 5 · Cancellation

A task can be cancelled with `task.cancel()`, which raises `asyncio.CancelledError` inside it at its next `await`. This is useful for timeouts and for abandoning work that is no longer needed. A coroutine can catch the cancellation to clean up, but should generally let it propagate.

In [ ]:
import asyncio

async def long_running():
    try:
        for i in range(100):
            await asyncio.sleep(0.01)
        return "completed"
    except asyncio.CancelledError:
        return "was cancelled"        # observe cancellation for cleanup, then re-raise normally

task = asyncio.create_task(long_running())
await asyncio.sleep(0.03)             # let it run a little
task.cancel()                         # request cancellation

try:
    await task
except asyncio.CancelledError:
    print("the task was cancelled as requested")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A simple coroutine (Easy)

In [ ]:
import asyncio

async def greet(name):
    await asyncio.sleep(0.01)
    return f"Hello, {name}"

print(await greet("Asha"))
# In a script this would be: print(asyncio.run(greet("Asha")))

### Example 2 — Two coroutines with gather (Easy)

In [ ]:
import asyncio

async def number(n):
    await asyncio.sleep(0.01)
    return n

results = await asyncio.gather(number(1), number(2), number(3))
print(results)

### Example 3 — Concurrency speed-up (Medium)

In [ ]:
import asyncio
import time

async def request(n):
    await asyncio.sleep(0.04)
    return n

start = time.perf_counter()
results = await asyncio.gather(*(request(n) for n in range(5)))
elapsed = time.perf_counter() - start

print("results:", results)
print(f"five 0.04s requests took about {elapsed:.2f}s (concurrent)")

### Example 4 — Background tasks (Medium)

In [ ]:
import asyncio

async def compute(label, delay, value):
    await asyncio.sleep(delay)
    return label, value

# Start two background tasks, then await both.
t1 = asyncio.create_task(compute("x", 0.03, 10))
t2 = asyncio.create_task(compute("y", 0.01, 20))

print("tasks started")
print(await t1)
print(await t2)

### Example 5 — A concurrent worker with bounded output (Difficult)

In [ ]:
import asyncio

async def process(item):
    await asyncio.sleep(0.02)
    return item.upper()

async def run_all(items):
    tasks = [asyncio.create_task(process(item)) for item in items]
    return await asyncio.gather(*tasks)

output = await run_all(["alpha", "beta", "gamma", "delta"])
print(output)

### Example 6 — A timeout using cancellation (Difficult)

In [ ]:
import asyncio

async def slow_operation():
    await asyncio.sleep(1.0)          # longer than we are willing to wait
    return "finished"

async def with_timeout(coro, seconds):
    task = asyncio.create_task(coro)
    try:
        return await asyncio.wait_for(task, timeout=seconds)   # cancels on timeout
    except asyncio.TimeoutError:
        return "timed out"

result = await with_timeout(slow_operation(), 0.05)
print("result:", result)

---

## Exercises

Use top-level `await` in your answers, as shown above (not `asyncio.run`, which does not work in a notebook).

**Exercise 1 (Easy).** Write a coroutine `double(n)` that awaits a short sleep and returns `n * 2`. Await it for `n = 21` and print the result.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `asyncio.gather` to run three calls of a coroutine `square(n)` for `n` in 2, 3, 4, and print the results.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a coroutine `fetch(name)` that sleeps briefly and returns a string. Use `gather` to fetch five names concurrently and print the list, noting it returns in passed order.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Start two background tasks with `create_task` (each returning a value after a short delay), do a small amount of other work, then await and print both results.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a coroutine that would take a long time, then use `asyncio.wait_for` with a short timeout to cancel it, returning the string `"timed out"` if it does not finish in time.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import asyncio

async def double(n):
    await asyncio.sleep(0.01)
    return n * 2

print(await double(21))

**Solution 2**

In [ ]:
import asyncio

async def square(n):
    await asyncio.sleep(0.01)
    return n * n

print(await asyncio.gather(square(2), square(3), square(4)))

**Solution 3**

In [ ]:
import asyncio

async def fetch(name):
    await asyncio.sleep(0.02)
    return f"data:{name}"

names = ["a", "b", "c", "d", "e"]
print(await asyncio.gather(*(fetch(n) for n in names)))

**Solution 4**

In [ ]:
import asyncio

async def work(label, delay):
    await asyncio.sleep(delay)
    return f"{label} done"

t1 = asyncio.create_task(work("first", 0.02))
t2 = asyncio.create_task(work("second", 0.01))
print("started")
print(await t1)
print(await t2)

**Solution 5**

In [ ]:
import asyncio

async def slow():
    await asyncio.sleep(1.0)
    return "done"

try:
    result = await asyncio.wait_for(slow(), timeout=0.05)
except asyncio.TimeoutError:
    result = "timed out"
print(result)

---

## Summary and Key Points

- `async def` defines a coroutine; calling it returns a coroutine object that an event loop must drive. `await` pauses until an operation completes, yielding control so other work proceeds.
- In a `.py` script the entry point is `asyncio.run(main())`; in a notebook a loop already runs, so use top-level `await` instead, as this notebook does.
- The event loop interleaves coroutines cooperatively at `await` points; one thing runs at a time, so there are no thread-style data races, but a coroutine that never awaits blocks everything.
- `asyncio.gather` runs coroutines concurrently and returns results in the order passed; for many overlapping waits this is highly efficient.
- `asyncio.create_task` schedules a coroutine to run in the background, returning a `Task` to await later; tasks can be cancelled with `cancel()` (or via `wait_for` timeouts), raising `CancelledError` inside them.

### Next module: 5.5 — Advanced Asyncio

The next module covers asynchronous context managers and iterators, async synchronisation primitives, structured concurrency with `TaskGroup`, and patterns for real input/output work.